In [ ]:
# Data is loaded directly from local files below — no upload needed

Saving spotify_recommendations.csv to spotify_recommendations.csv
Saving spotify_history.csv to spotify_history.csv


In [ ]:
"""
Per-play skip prediction — with ablations
=========================================
Trains three variants of the same model on the same temporal split, so you
can see what each feature group is actually contributing:

  1. Full model        — all context features.
  2. No reason_start   — strips the "previous play's end → this play's start"
                          signal. Isolates how much the model relies on
                          immediate session state.
  3. Pure context      — also strips minutes_since_last. What's left is the
                          underlying behavioral pattern: time of day, day of
                          week, month, platform, shuffle. The "when do I
                          skip" model.
"""

import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBClassifier


# ── 1. Load ───────────────────────────────────────────────────────────────────

history = pd.read_csv("../data/cleaned_streaming_history.csv", parse_dates=["ts"])
print(f"Loaded {len(history):,} plays  "
      f"({history['ts'].min().date()} → {history['ts'].max().date()})")


# ── 2. Build label ────────────────────────────────────────────────────────────

SKIP_END_REASONS = {"fwdbtn", "backbtn"}
history["is_skip"] = (
    history["skipped"].fillna(False).astype(bool) |
    history["reason_end"].isin(SKIP_END_REASONS)
).astype(int)

print(f"Overall skip rate: {history['is_skip'].mean():.3f}")


# ── 3. Build context features ─────────────────────────────────────────────────
# Hard rule: features describe THE MOMENT the play started.
#   - No ms_played / seconds_played / minutes_played  (these are outcomes)
#   - No likely_skipped                              (derived from outcome)
#   - No track-level aggregates                      (bake the label in)

df = history.sort_values("ts").reset_index(drop=True).copy()

# Time since previous play (in minutes, capped at 24h).
df["minutes_since_last"] = (
    df["ts"].diff().dt.total_seconds().div(60).fillna(0).clip(upper=24 * 60)
)

# Cyclical time encodings.
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

dow_map = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
           "Friday": 4, "Saturday": 5, "Sunday": 6}
df["dow_num"] = df["day_of_week"].map(dow_map)
df["dow_sin"] = np.sin(2 * np.pi * df["dow_num"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow_num"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["shuffle"]    = df["shuffle"].astype(int)
df["is_weekend"] = df["dow_num"].isin([5, 6]).astype(int)

for col in ["platform", "reason_start"]:
    df[col] = df[col].fillna("unknown").astype("category")


# ── 4. Define feature sets for each ablation ──────────────────────────────────

PURE_CONTEXT = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos", "is_weekend",
    "month_sin", "month_cos",
    "shuffle",
    "platform",
]

VARIANTS = {
    "full":             PURE_CONTEXT + ["minutes_since_last", "reason_start"],
    "no_reason_start":  PURE_CONTEXT + ["minutes_since_last"],
    "pure_context":     PURE_CONTEXT,
}


# ── 5. Temporal split ─────────────────────────────────────────────────────────

CUTOFF = pd.Timestamp("2024-01-01")
train_mask = df["ts"] <  CUTOFF
test_mask  = df["ts"] >= CUTOFF

y_train = df.loc[train_mask, "is_skip"]
y_test  = df.loc[test_mask,  "is_skip"]

print(f"\nTrain: {len(y_train):,}  "
      f"({df.loc[train_mask,'ts'].min().date()} → {df.loc[train_mask,'ts'].max().date()})")
print(f"Test : {len(y_test):,}  "
      f"({df.loc[test_mask, 'ts'].min().date()} → {df.loc[test_mask, 'ts'].max().date()})")
print(f"Train skip rate: {y_train.mean():.3f}")
print(f"Test  skip rate: {y_test.mean():.3f}")

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
print(f"scale_pos_weight: {scale_pos_weight:.3f}  (neg={neg:,}, pos={pos:,})")


# ── 6. Train + evaluate each variant ──────────────────────────────────────────

def train_and_eval(features, name):
    X_train = df.loc[train_mask, features]
    X_test  = df.loc[test_mask,  features]

    model = XGBClassifier(
        n_estimators       = 300,
        max_depth          = 5,
        learning_rate      = 0.05,
        subsample          = 0.8,
        colsample_bytree   = 0.8,
        scale_pos_weight   = scale_pos_weight,
        eval_metric        = "logloss",
        enable_categorical = True,
        tree_method        = "hist",
        random_state       = 42,
        verbosity          = 0,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "variant":   name,
        "n_features": len(features),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall":    recall_score(y_test, y_pred, zero_division=0),
        "f1":        f1_score(y_test, y_pred, zero_division=0),
        "auc":       roc_auc_score(y_test, y_prob),
    }

    print(f"\n── {name.upper()}  ({len(features)} features) ──")
    print(classification_report(y_test, y_pred,
                                target_names=["Not Skipped", "Skipped"]))
    print(f"Precision={metrics['precision']:.4f}  "
          f"Recall={metrics['recall']:.4f}  "
          f"F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc']:.4f}")
    print("Confusion matrix:")
    print(pd.DataFrame(confusion_matrix(y_test, y_pred),
                       index=["Not Skipped", "Skipped"],
                       columns=["Not Skipped", "Skipped"]).to_string())
    print("Top 5 features by importance:")
    imp = pd.DataFrame({
        "feature":    features,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).head(5)
    print(imp.to_string(index=False))

    return metrics


results = [train_and_eval(feats, name) for name, feats in VARIANTS.items()]


# ── 7. Comparison table ───────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("VARIANT COMPARISON")
print("=" * 60)
comparison = pd.DataFrame(results)[
    ["variant", "n_features", "precision", "recall", "f1", "auc"]
]
print(comparison.to_string(index=False))

Loaded 145,139 plays  (2013-07-08 → 2024-12-15)
Overall skip rate: 0.377

Train: 135,282  (2013-07-08 → 2023-12-31)
Test : 9,857  (2024-01-01 → 2024-12-15)
Train skip rate: 0.388
Test  skip rate: 0.222
scale_pos_weight: 1.576  (neg=82,757, pos=52,525)

── FULL  (11 features) ──
              precision    recall  f1-score   support

 Not Skipped       0.94      0.98      0.96      7667
     Skipped       0.92      0.78      0.85      2190

    accuracy                           0.94      9857
   macro avg       0.93      0.88      0.90      9857
weighted avg       0.94      0.94      0.93      9857

Precision=0.9237  Recall=0.7799  F1=0.8458  AUC=0.9550
Confusion matrix:
             Not Skipped  Skipped
Not Skipped         7526      141
Skipped              482     1708
Top 5 features by importance:
           feature  importance
minutes_since_last    0.503578
      reason_start    0.285450
           shuffle    0.108462
         month_sin    0.022043
          hour_sin    0.013357

──